# Импорты

In [43]:
import plotly.express as px
import numpy as np
import sympy as sp
from scipy.integrate import quad

# Входные данные

Уравнение:<br>
$
\begin{cases} \displaystyle
    Lu \equiv u'' - \frac{3}{(x + 1)^2} u = \frac{-1,5}{\sqrt{x + 1}}, ~~~ 0 < x < 1 \\
    \displaystyle l_0 u \equiv u(0) = \frac{2}{3} \\
    \displaystyle  l_1 u \equiv u(1) = \frac{4 \sqrt{2}}{3}
\end{cases}
$

Решение:<br>
$\displaystyle
u(x) = \frac{2}{3} (x + 1)^{\frac{3}{2}}$

In [44]:
n = 3

x = sp.Symbol("x")
u = sp.Function("u")

L = u(x).diff(x, 2) - 3 / (x + 1) ** 2 * u(x)
l_0 = u(x).subs(x, 0)
l_1 = u(x).subs(x, 1)

P = 0 * x
Q = -3 / (x + 1) ** 2
F = -1.5 / sp.sqrt(x + 1)

gamma_0 = 1
gamma_1 = 2 ** 0.5

u_exact = 2 / 3 * (x + 1) ** 1.5

# Решение

$\displaystyle
u(x) = \varphi_0(x) + \sum_{i = 1}^{n} C_i \varphi_i(x) \\
$, где $C_i -$ неизвестные коэффициенты, $\varphi_0, \varphi_1, \ldots, \varphi_n, \ldots -$ полная ортогональная система базисных функций на отрезке [0, 1], для которых выполняется: <br>
$
l_0 \varphi_0 = 3 \varphi_0(0) - \varphi'_0(0) = 1 \\
l_1 \varphi_0 = \varphi'_0 (1) = \sqrt{2} \\
l_0 \varphi_i = l_1 \varphi_i = 0, ~~ i = \overline{1, n}
$

Определим дискретно-непрерывное скалярное произведение типа Соболева: <br>
$\displaystyle (f, g)_S = f(0)g(0) + f(1)g(1) + \int_0^1 f'(x) g'(x) dx$

Для этого скалярного произведения найдём систему $\varphi_0, \varphi_1, \ldots, \varphi_n, \ldots \\$
$\displaystyle
\varphi_0 = \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \\
\varphi_i = \sin(\pi i x), ~~ i = 1, 2, \ldots \\
$
Проверим выполнение условий:
1) Соблюдение граничный условий: <br>
    * $\displaystyle l_0 \varphi_0 = \varphi_0(0) = \frac{2}{3} $
    * $\displaystyle l_1 \varphi_0 = \varphi_0(1) = \frac{4\sqrt{2}}{3} $
    * $\displaystyle l_0 \varphi_i = \sin(0) = 0, ~~ i = 1, 2, \ldots $
    * $\displaystyle l_1 \varphi_i = \sin(\pi i) = 0, ~~ i = 1, 2, \ldots $
2) Линейная независимость: <br>
    * Функции $\sin(\pi k x)$ составляют ортогональный базис
    * $\varphi_0$ является полиномом, поэтому относительно функций синусов он также линейно независим
3) Ортогональность: <br>
    * $\displaystyle
    (\varphi_k, \varphi_m) = (\sin(\pi k x), \sin(\pi m x)) = \sin(0) \sin(0) + \sin(\pi k) \sin(\pi m) + \int_0^1 \sin'(\pi k x) \sin'(\pi m x) dx = km \pi^2 \int_0^1 \cos(\pi k x) \cos(\pi m x) dx = \frac{km \pi^2}{2} \left( \int_0^1 cos((k + m) \pi x) dx + \int_0^1 cos((k - m) \pi x) dx \right) = \frac{km \pi^2}{2} \left( \frac{1}{(k + m) \pi} sin((k + m) \pi x) \Big|_0^1 + \frac{1}{(k - m) \pi} sin((k - m) \pi x) \Big|_0^1 \right) = 0, ~~ k \neq m
    $
    * $\displaystyle
    (\varphi_0, \varphi_i) = \left( \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3}, \sin(\pi i x) \right) = \frac{2}{3} \sin(0) + \frac{4\sqrt{2}}{3} \sin(\pi i) + \int_0^1 \left( \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \right)' \sin'(\pi i x) dx = \frac{(4\sqrt{2} - 2)\pi i}{3} \int_0^1 \cos(\pi i x) dx = \frac{4\sqrt{2} - 2}{3} \sin(\pi i x) \Big|_0^1 = 0, ~~ i = 1, 2, \ldots
    $
4) Полнота: <br>
    * Исходя из определения скалярного произведения, ортогональной к функции $\varphi_i, i = 1, 2, \ldots$ будет любой полином 1-й и 0-й степеней
    * Однако ни один полином 1-й или 0-й степени не будет ортогонален $\varphi_0$, кроме 0

In [45]:
k = sp.Symbol("k")

phi_0 = (sp.Integer(4) * sp.sqrt(2) - 2) / 3 * x + sp.Rational(2, 3)
phi_k = sp.sin(k * sp.pi * x)

## Метод коллокации

$\displaystyle
R(x, C_1, \ldots, C_n) = Lu(x) - f(x) = L\varphi_0(x) - f(x) + \sum_{i = 1}^n C_i L\varphi_i(x) = \frac{1,5}{\sqrt{x + 1}} -\frac{3}{(x + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \right) - \sum_{i = 1}^n C_i \left( \pi^2 i^2 \sin(\pi i x) + \frac{3}{(x + 1)^2} \sin(\pi i x) \right) = 0 \\
x = x_1, \ldots, x_n \in [0, 1] \\
\left(\begin{array}{cccc}
    \displaystyle \pi^2 \sin(\pi x_1) + \frac{3}{(x_1 + 1)^2} \sin(\pi x_1) & \displaystyle 4\pi^2 \sin(2\pi x_1) + \frac{3}{(x_1 + 1)^2} \sin(2\pi x_1) & \ldots & \displaystyle n^2 \pi^2 \sin(n\pi x_1) + \frac{3}{(x_1 + 1)^2} \sin(n\pi x_1) \\
    \displaystyle \pi^2 \sin(\pi x_2) + \frac{3}{(x_2 + 1)^2} \sin(\pi x_2) & \displaystyle 4\pi^2 \sin(2\pi x_2) + \frac{3}{(x_2 + 1)^2} \sin(2\pi x_2) & \ldots & \displaystyle n^2 \pi^2 \sin(n\pi x_2) + \frac{3}{(x_2 + 1)^2} \sin(n\pi x_2) \\
    \vdots & \vdots & \ddots & \vdots \\
    \displaystyle \pi^2 \sin(\pi x_n) + \frac{3}{(x_n + 1)^2} \sin(\pi x_n) & \displaystyle 4\pi^2 \sin(2\pi x_n) + \frac{3}{(x_n + 1)^2} \sin(2\pi x_n) & \ldots & \displaystyle n^2 \pi^2 \sin(n\pi x_n) + \frac{3}{(x_n + 1)^2} \sin(n\pi x_n) \\
\end{array}\right) \left(\begin{array}{c}
    C_1 \\ C_2 \\ \vdots \\ C_n
\end{array}\right) = \left(\begin{array}{c}
    \displaystyle \frac{1,5}{\sqrt{x_1 + 1}} - \frac{3}{(x_1 + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x_1 + \frac{2}{3} \right) \\
    \displaystyle \frac{1,5}{\sqrt{x_2 + 1}} - \frac{3}{(x_2 + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x_2 + \frac{2}{3} \right) \\
    \vdots \\
    \displaystyle \frac{1,5}{\sqrt{x_n + 1}} - \frac{3}{(x_n + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x_n + \frac{2}{3} \right) \\
\end{array}\right)
$

In [46]:
def method_of_collocations() -> sp.core.add.Add:
    x_i = np.linspace(0, 1, n + 2)[1 : -1]

    A = np.zeros((n, n))
    b = np.zeros(n)

    for i in range(n):
        for j in range(n):
            L_phi_j = L.subs(u(x), phi_k.subs(k, j + 1))
            A[i, j] = L_phi_j.subs(x, x_i[i]).doit()
        
        L_phi_0 = L.subs(u(x), phi_0)
        b[i] = L_phi_0.subs(x, x_i[i]).doit() - F.subs(x, x_i[i]) 

    C = np.linalg.solve(A, b)
    u_x = phi_0 + sum(C[i] * phi_k.subs(k, i + 1) for i in range(n))
    
    return u_x

u_collocations = method_of_collocations()
u_collocations

x*(-2/3 + 4*sqrt(2)/3) + 0.0505920280373601*sin(pi*x) + 0.000909226310756371*sin(2*pi*x) + 0.000998239479320794*sin(3*pi*x) + 2/3

## Метод Ритца

$\displaystyle
p(x) = e^{\scriptsize{\displaystyle \int_0^x P(x) dx}} = e^{\scriptsize{\displaystyle \int_0^x 0 dx}} = e^0 = 1 \\
q(x) = p(x) Q(x) = -\frac{3}{(x + 1)^2} \\
f(x) = p(x) F(x) = \frac{-1,5}{\sqrt{x + 1}}
$

Функционал: <br>
$\displaystyle
J[u] = \Phi(C_1, \ldots, C_n) = \int_0^1 \left( \left( \varphi'_0(x) + \sum_{i = 1}^n C_i \varphi'_i(x) \right)^2 + \frac{3}{(x + 1)^2} \left( \varphi_0(x) + \sum_{i = 1}^n C_i \varphi_i(x) \right)^2 - \frac{3}{\sqrt{x + 1}} \left( \varphi_0(x) + \sum_{i = 1}^n C_i \varphi_i(x) \right) \right) dx
$

Из этого функционала выводятся условия: <br>
$\displaystyle
\frac{\partial \Phi}{\partial C_i} = 0, ~~ i = \overline{1, n} \\
\frac{\partial \Phi}{\partial C_i} = \int_0^1 \left( 2 \left( \varphi'_0(x) + \sum_{i = 1}^n C_i \varphi'_i(x) \right) \varphi'_i(x) + \frac{6}{(x + 1)^2} \left( \varphi_0(x) + \sum_{i = 1}^n C_i \varphi_i(x) \right) \varphi_i(x) - \frac{3}{\sqrt{x + 1}} \varphi_i(x) \right) dx = 0 \\
\int_0^1 \left( 2 \left( \varphi'_0(x) + \sum_{j = 1}^n C_j \varphi'_j(x) \right) \varphi'_i(x) + \frac{6}{(x + 1)^2} \left( \varphi_0(x) + \sum_{j = 1}^n C_j \varphi_j(x) \right) \varphi_i(x) \right) dx = \int_0^1 \frac{3}{\sqrt{x + 1}} \varphi_i(x) dx \\
\int_0^1 \left( 2\varphi'_i(x) \sum_{j = 1}^n C_j \varphi'_j(x) + \frac{6}{(x + 1)^2} \varphi_i(x) \sum_{j = 1}^n C_j \varphi_j(x) \right) dx = \int_0^1 \left( \frac{3}{\sqrt{x + 1}} \varphi_i(x) - 2\varphi'_0(x) \varphi'_i(x) - \frac{6}{(x + 1)^2} \varphi_0(x) \varphi_i(x) \right) dx \\
\sum_{j = 1}^n C_j \int_0^1 \left( 2\varphi'_j(x) \varphi'_i(x) + \frac{6}{(x + 1)^2} \varphi_j(x) \varphi_i(x) \right) dx = \int_0^1 \left( \frac{3}{\sqrt{x + 1}} \varphi_i(x) - 2\varphi'_0(x) \varphi'_i(x) - \frac{6}{(x + 1)^2} \varphi_0(x) \varphi_i(x) \right) dx \\
\left(\begin{array}{cccc}
    \displaystyle \int_0^1 \left( 2\varphi'_1(x) \varphi'_1(x) + \frac{6}{(x + 1)^2} \varphi_1(x) \varphi_1(x) \right) dx & \displaystyle \int_0^1 \left( 2\varphi'_2(x) \varphi'_1(x) + \frac{6}{(x + 1)^2} \varphi_2(x) \varphi_1(x) \right) dx & \ldots & \displaystyle \int_0^1 \left( 2\varphi'_n(x) \varphi'_1(x) + \frac{6}{(x + 1)^2} \varphi_n(x) \varphi_1(x) \right) dx \\
    \displaystyle \int_0^1 \left( 2\varphi'_1(x) \varphi'_2(x) + \frac{6}{(x + 1)^2} \varphi_1(x) \varphi_2(x) \right) dx & \displaystyle \int_0^1 \left( 2\varphi'_2(x) \varphi'_2(x) + \frac{6}{(x + 1)^2} \varphi_2(x) \varphi_2(x) \right) dx & \ldots & \displaystyle \int_0^1 \left( 2\varphi'_n(x) \varphi'_2(x) + \frac{6}{(x + 1)^2} \varphi_n(x) \varphi_2(x) \right) dx \\
    \vdots & \vdots & \ddots & \vdots \\
    \displaystyle \int_0^1 \left( 2\varphi'_1(x) \varphi'_n(x) + \frac{6}{(x + 1)^2} \varphi_1(x) \varphi_n(x) \right) dx & \displaystyle \int_0^1 \left( 2\varphi'_2(x) \varphi'_n(x) + \frac{6}{(x + 1)^2} \varphi_2(x) \varphi_n(x) \right) dx & \ldots & \displaystyle \int_0^1 \left( 2\varphi'_n(x) \varphi'_n(x) + \frac{6}{(x + 1)^2} \varphi_n(x) \varphi_n(x) \right) dx
\end{array}\right) \cdot \left( \begin{array}{c}
    C_1 \\ C_2 \\ \vdots \\ C_n
\end{array} \right) = \left( \begin{array}{c}
    \displaystyle \int_0^1 \left( \frac{3}{\sqrt{x + 1}} \varphi_1(x) - 2\varphi'_0(x) \varphi'_1(x) - \frac{6}{(x + 1)^2} \varphi_0(x) \varphi_1(x) \right) dx \\
    \displaystyle \int_0^1 \left( \frac{3}{\sqrt{x + 1}} \varphi_2(x) - 2\varphi'_0(x) \varphi'_2(x) - \frac{6}{(x + 1)^2} \varphi_0(x) \varphi_2(x) \right) dx \\
    \vdots \\
    \displaystyle \int_0^1 \left( \frac{3}{\sqrt{x + 1}} \varphi_n(x) - 2\varphi'_0(x) \varphi'_n(x) - \frac{6}{(x + 1)^2} \varphi_0(x) \varphi_n(x) \right) dx
\end{array} \right)
$

In [47]:
def ritz_method() -> sp.core.add.Add:
    t = sp.Symbol("t")
    p = sp.exp(sp.integrate(P, (x, 0, t))).subs(t, x)
    q = p * Q
    f = p * F

    A = np.zeros((n, n))
    b = np.zeros(n)

    for i in range(n):
        phi_i = phi_k.subs(k, i + 1)
        a = -2 * f * phi_i - 2 * p * phi_0.diff(x) * phi_i.diff(x) + 2 * q * phi_0 * phi_i
        b[i] = quad(sp.lambdify(x, a), 0, 1, epsrel=1e-10)[0]

        for j in range(i, n):
            phi_j = phi_k.subs(k, j + 1)
            a = 2 * p * phi_j.diff(x) * phi_i.diff(x) - 2 * q * phi_j * phi_i
            A[i, j] = quad(sp.lambdify(x, a), 0, 1, epsabs=1e-10)[0]
            A[j, i] = A[i, j]

    C = np.linalg.solve(A, b)
    u_x = phi_0 + sum(C[i] * phi_k.subs(k, i + 1) for i in range(n))

    return u_x


u_ritz = ritz_method()
u_ritz

x*(-2/3 + 4*sqrt(2)/3) - 0.0530983033806421*sin(pi*x) - 0.00112956111260387*sin(2*pi*x) - 0.00202931859346371*sin(3*pi*x) + 2/3

## Метод Галёркина

Невязка: <br>
$\displaystyle
R(x, C_1, \ldots, C_n) = L \left( \varphi_0(x) + \sum_{j = 1}^n C_j \varphi_j(x) \right) - f(x) \\
(R, \varphi_i) = \int_0^1 \varphi_i(x) R(x, C_1, \ldots, C_n) dx = 0, ~~ i = \overline{1, n}
$

Из невязки получаем систему: <br>
$\displaystyle
\left( \begin{array}{cccc}
    \displaystyle \int_0^1 \varphi_1(x) L\varphi_1(x) dx & \displaystyle \int_0^1 \varphi_1(x) L\varphi_2(x) dx & \ldots & \displaystyle \int_0^1 \varphi_1(x) L\varphi_n(x) dx \\
    \displaystyle \int_0^1 \varphi_2(x) L\varphi_1(x) dx & \displaystyle \int_0^1 \varphi_2(x) L\varphi_2(x) dx & \ldots & \displaystyle \int_0^1 \varphi_2(x) L\varphi_n(x) dx \\
    \vdots & \vdots & \ddots & \vdots \\
    \displaystyle \int_0^1 \varphi_n(x) L\varphi_1(x) dx & \displaystyle \int_0^1 \varphi_n(x) L\varphi_2(x) dx & \ldots & \displaystyle \int_0^1 \varphi_n(x) L\varphi_n(x) dx \\
\end{array} \right) \left( \begin{array}{c}
    C_1 \\ C_2 \\ \vdots \\ C_n
\end{array} \right) = \left( \begin{array}{c}
    \displaystyle \int_0^1 \varphi_1(x) (f(x) - L\varphi_0(x)) dx \\
    \displaystyle \int_0^1 \varphi_2(x) (f(x) - L\varphi_0(x)) dx \\
    \vdots \\
    \displaystyle \int_0^1 \varphi_n(x) (f(x) - L\varphi_0(x)) dx
\end{array} \right)
$

In [48]:
def galerkin_method() -> sp.core.add.Add:
    L_phi_0 = L.subs(u(x), phi_0)

    A = np.zeros((n, n))
    b = np.zeros(n)

    for i in range(n):
        phi_i = phi_k.subs(k, i + 1)
        a = sp.lambdify(x, (phi_i * (F - L_phi_0)).doit())
        b[i] = quad(a, 0, 1)[0]

        for j in range(n):
            L_phi_j = L.subs(u(x), phi_k.subs(k, j + 1))
            a = sp.lambdify(x, (phi_i * L_phi_j).doit())
            A[i, j] = quad(a, 0, 1)[0]

    C = np.linalg.solve(A, b)
    u_x = phi_0 + sum(C[i] * phi_k.subs(k, i + 1) for i in range(n))

    return u_x

u_galerkin = galerkin_method()
u_galerkin

x*(-2/3 + 4*sqrt(2)/3) - 0.0530983033806421*sin(pi*x) - 0.00112956111260387*sin(2*pi*x) - 0.00202931859346371*sin(3*pi*x) + 2/3

## Метод конечных элементов

При неоднородных краевых условиях: <br>
$\displaystyle
u(0) = \frac{2}{3} \\
u(1) = \frac{4 \sqrt{2}}{3} \\
$
Сводим задачу к: <br>
$\displaystyle
L\omega(x) = f_{\omega}(x) \\
v(x) = \frac{2}{3} + \left( \frac{4 \sqrt{2}}{3} - \frac{2}{3} \right) x = \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \\
f_{\omega} = f(x) - p(x) v'(x) - q(x) v(x) = \frac{-1,5}{\sqrt{x + 1}} - \frac{3}{(x + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \right)  \\
\omega(0) = \omega(1) = 0 \\
\omega_n(x) = \sum_{i = 0}^n C_i \varphi_i(x) \\
u(x) \approx \omega_n(x) + v(x) \\
$

Базис: <br>
$\displaystyle
h = \frac{1}{n + 1} \\
x_i = ih, ~~ i = \overline{0, n + 1} \\
\varphi_i = \begin{cases}
    \displaystyle \frac{x - x_{i - 1}}{h}, & x \in [x_{i - 1}; x_i] \\
    \displaystyle -\frac{x - x_{i + 1}}{h}, & x \in [x_i; x_{i + 1}] \\
    0, & x \notin [x_{i - 1}; x_{i + 1}]
\end{cases}
$

Система: <br>
$\displaystyle
A_{ii} = -\frac{2}{h} - \frac{1}{h^2} \left( \int_{x_{i - 1}}^{x_i} \frac{3}{(x + 1)^2} (x - x_{i - 1})^2 dx + \int_{x_i}^{x_{i + 1}} \frac{3}{(x + 1)^2} (x - x_{i + 1})^2 dx \right) \\
A_{i, i + 1} = \frac{1}{h} + \frac{1}{h^2} \int_{x_i}^{x_{i + 1}} \frac{3}{(x + 1)^2} (x - x_i) (x - x_{i + 1}) dx \\
A_{i, i - 1} = \frac{1}{h} + \frac{1}{h^2} \int_{x_{i - 1}}^{x_i} \frac{3}{(x + 1)^2} (x - x_{i - 1}) (x - x_i) dx \\
b_i = \frac{1}{h} \left( \int_{x_{i - 1}}^{x_i} \left( \frac{-1,5}{\sqrt{x + 1}} - \frac{3}{(x + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \right) \right) (x - x_{i - 1}) dx - \int_{x_i}^{x_{i + 1}} \left( \frac{-1,5}{\sqrt{x + 1}} - \frac{3}{(x + 1)^2} \left( \frac{4\sqrt{2} - 2}{3} x + \frac{2}{3} \right) \right) (x - x_{i + 1}) dx \right) \\
\left( \begin{array}{cccccccc}
    A_{11} & A_{12} & 0 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    A_{21} & A_{22} & A_{23} & 0 & \ldots & 0 & 0 & 0 & 0 \\
    0 & A_{32} & A_{33} & A_{34} & \ldots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \ldots & A_{n - 2, n - 3} & A_{n - 2, n - 2} & A_{n - 2, n - 1} & 0 \\
    0 & 0 & 0 & 0 & \ldots & 0 & A_{n - 1, n - 2} & A_{n - 1, n - 1} & A_{n - 1, n} \\
    0 & 0 & 0 & 0 & \ldots & 0 & 0 & A_{n, n - 1} & A_{n, n} \\
\end{array} \right) \cdot \left( \begin{array}{c}
    C_1 \\ C_2 \\ C_3 \\ \vdots \\ C_{n - 2} \\ C_{n - 1} \\ C_n
\end{array} \right) = \left( \begin{array}{c}
    b_1 \\ b_2 \\ b_3 \\ \vdots \\ b_{n - 2} \\ b_{n - 1} \\ b_n
\end{array} \right)
$

In [49]:
def finite_element_method() -> sp.core.add.Add:
    h = sp.Rational(1, n + 1)
    x_i = [h * i for i in range(n + 2)]

    f_omega = F - Q * phi_0

    A = np.zeros(n)
    B = np.zeros(n)
    C = np.zeros(n)
    b = np.zeros(n)

    for i in range(1, n + 1):
        if i > 1:
            a = sp.integrate(Q * (x - x_i[i - 1]) * (x - x_i[i]), (x, x_i[i - 1], x_i[i]))
            A[i - 1] = float((1 / h - 1 / h ** 2 *  a).doit())
        
        b_1 = sp.integrate(Q * (x - x_i[i - 1]) ** 2, (x, x_i[i - 1], x_i[i]))
        b_2 = sp.integrate(Q * (x - x_i[i + 1]) ** 2, (x, x_i[i], x_i[i + 1]))
        B[i - 1] = float((-2 / h + 1 / h ** 2 * (b_1 + b_2)).doit())

        if i < n:
            c = sp.integrate(Q * (x - x_i[i]) * (x - x_i[i + 1]), (x, x_i[i], x_i[i + 1]))
            C[i - 1] = float((1 / h - 1 / h ** 2 * c).doit())
        
        b_1 = quad(sp.lambdify(x, f_omega * (x - x_i[i - 1])), x_i[i - 1], x_i[i])[0]
        b_2 = quad(sp.lambdify(x, f_omega * (x - x_i[i + 1])), x_i[i], x_i[i + 1])[0]
        b[i - 1] = 1 / h * (b_1 + b_2)


    def monotonous_reduction(A: np.array, B: np.array, C: np.array, F: np.array) -> np.array:
        n = len(A) - 1

        alpha = np.zeros(n + 1)
        beta = np.zeros(n + 1)
        U = np.zeros(n + 1)

        alpha[1] = -C[0] / B[0]
        beta[1] = F[0] / B[0]

        for i in range(1, n):
            alpha[i + 1] = -C[i] / (B[i] + alpha[i] * A[i])
            beta[i + 1] = (F[i] - beta[i] * A[i]) / (B[i] + alpha[i] * A[i])

        U[n] = (F[n] - beta[n] * A[n]) / (B[n] + alpha[n] * A[n])

        for i in range(n - 1, -1, -1):
            U[i] = alpha[i + 1] * U[i + 1] + beta[i + 1]

        return U

    def basis(i: int, x_i: list, h: sp.Rational) -> sp.Piecewise:
        xi = x_i[i]
        xi_1 = x_i[i - 1]
        xi_2 = x_i[i + 1]

        return sp.Piecewise(
            ((x - xi_1) / h, (x >= xi_1) & (x <= xi)),
            (-(x - xi_2) / h, (x >= xi) & (x <= xi_2)),
            (0, True)
        )
    

    C = monotonous_reduction(A, B, C, b)
    u_x = phi_0
    for i in range(1, n + 1):
        u_x += C[i - 1] * basis(i, x_i, h)

    return u_x

u_finite_element = finite_element_method()
u_finite_element

x*(-2/3 + 4*sqrt(2)/3) - 0.00109508620113835*Piecewise((4*x, (x >= 0) & (x <= 1/4)), (2 - 4*x, (x >= 1/4) & (x <= 1/2)), (0, True)) - 0.00153037242522522*Piecewise((4*x - 2, (x >= 1/2) & (x <= 3/4)), (4 - 4*x, (x >= 3/4) & (x <= 1)), (0, True)) - 0.0018420655974004*Piecewise((4*x - 1, (x >= 1/4) & (x <= 1/2)), (3 - 4*x, (x >= 1/2) & (x <= 3/4)), (0, True)) + 2/3

# Визуализация

In [50]:
def u_i(x_i: np.array, u: sp.add.Add) -> float:
    return [float(u.subs(x, x_i[i])) for i in range(len(x_i))]

## Сравнение графиков

In [51]:
x_i = np.linspace(0, 1, 100)

figure = px.line(x=x_i, y=u_i(x_i, u_exact), markers=False)
figure.update_traces(name="Точное решение", showlegend=True)

figure.add_scatter(x=x_i, y=u_i(x_i, u_collocations), name="Решение методом коллокаций")
figure.add_scatter(x=x_i, y=u_i(x_i, u_ritz), name="Решение методом Ритца")
figure.add_scatter(x=x_i, y=u_i(x_i, u_galerkin), name="Решение методом Галёркина")
figure.add_scatter(x=x_i, y=u_i(x_i, u_finite_element), name="Решение методом конечных элементов")

figure.show()

## Погрешность

In [52]:
figure = px.line(x=x_i, y=u_i(x_i, u_exact), markers=False)
figure.update_traces(name="Точное решение", showlegend=True)
figure.update_layout(yaxis=dict(exponentformat="power"))

figure.add_scatter(x=x_i, y=u_i(x_i, sp.Abs(u_exact - u_collocations)), name="Погрешность метода коллокаций")
figure.add_scatter(x=x_i, y=u_i(x_i, sp.Abs(u_exact - u_ritz)), name="Погрешность метода Ритца")
figure.add_scatter(x=x_i, y=u_i(x_i, sp.Abs(u_exact - u_galerkin)), name="Погрешность метода Галёркина")
figure.add_scatter(x=x_i, y=u_i(x_i, sp.Abs(u_exact - u_finite_element)), name="Погрешность метода конечных элементов")

figure.show()